In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
from flask import Flask, render_template_string

# 解决中文显示乱码
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 初始化Flask
app = Flask(__name__)

# -------------------------- 完全复用你的配置 --------------------------
PHOTOS_FOLDER = r"D:\bishe\esp32_photos_auto"
MODEL_PATH = r"D:\86151\bishedaima\disease_model_optimized.h5"
CONFIDENCE_THRESHOLD = 50
IMAGE_SIZE = (128, 128)
CLASS_NAMES_CN = [
    '番茄细菌性斑点病', '番茄早疫病', '番茄健康叶片',
    '番茄晚疫病', '番茄叶霉病', '番茄壳针孢叶斑病',
    '番茄二斑叶螨虫害', '番茄靶斑病', '番茄花叶病毒病',
    '番茄黄化曲叶病毒病'
]

# -------------------------- 完全复用你的识别逻辑 --------------------------
def do_recognize():
    # 1. 找最新照片
    if not os.path.exists(PHOTOS_FOLDER):
        return None, "文件夹不存在，请先创建文件夹"
    
    image_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    all_image_files = [
        os.path.join(PHOTOS_FOLDER, filename)
        for filename in os.listdir(PHOTOS_FOLDER)
        if filename.endswith(image_extensions)
    ]
    
    if len(all_image_files) == 0:
        return None, "文件夹里没有照片，请先从ESP32下载照片"
    
    all_image_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    latest_image_path = all_image_files[0]
    image_name = os.path.basename(latest_image_path)

    # 2. 读取图片
    img_bgr = cv2.imread(latest_image_path)
    if img_bgr is None:
        from PIL import Image
        try:
            img_original = Image.open(latest_image_path).convert('RGB')
            img_rgb = np.array(img_original)
        except:
            return None, "图片文件损坏"
    else:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 预处理+预测
    img_resized = cv2.resize(img_rgb, IMAGE_SIZE)
    img_normalized = img_resized / 255.0
    model = tf.keras.models.load_model(MODEL_PATH)
    img_input = np.expand_dims(img_normalized, axis=0)
    pred_prob = model.predict(img_input, verbose=0)[0]
    pred_class_idx = np.argmax(pred_prob)
    pred_class_name = CLASS_NAMES_CN[pred_class_idx]
    pred_confidence = round(float(pred_prob[pred_class_idx] * 100), 2)

    # 4. 返回结果
    result = {
        'name': image_name,
        'class': pred_class_name,
        'confidence': pred_confidence,
        'is_healthy': pred_class_name == '番茄健康叶片',
        'warning': '叶片健康，无需处理' if pred_class_name == '番茄健康叶片' else f'检测到【{pred_class_name}】，建议及时摘除病叶并喷施对应药剂'
    }
    return result, None

# -------------------------- 网页页面 --------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>番茄病虫害AI识别</title>
    <style>
        body {max-width: 800px; margin: 0 auto; padding: 40px 20px; font-family: "SimHei", sans-serif;}
        h1 {text-align: center; margin-bottom: 30px;}
        .btn {padding: 15px 40px; border: none; border-radius: 8px; color: white; font-size: 18px; cursor: pointer; background: #3498db; display: block; margin: 0 auto;}
        .btn:hover {background: #2980b9;}
        .result-box {margin-top: 40px; padding: 30px; border-radius: 12px; background: #f8f9fa;}
        .result-title {font-size: 24px; font-weight: bold; text-align: center; margin-bottom: 20px;}
        .success {color: #27ae60;}
        .danger {color: #dc3545;}
        .info {color: #333; margin-top: 10px; text-align: center; font-size: 16px;}
    </style>
</head>
<body>
    <h1>番茄叶片病虫害AI识别</h1>
    <button class="btn" onclick="location.reload()">🔄 点击识别最新照片</button>
    
    {% if result %}
    <div class="result-box">
        <div class="result-title {{ 'success' if result.is_healthy else 'danger' }}">
            识别结果：{{ result.class }} | 置信度：{{ result.confidence }}%
        </div>
        <div class="info">识别照片：{{ result.name }}</div>
        <div class="info" style="margin-top: 20px; font-weight: bold;">{{ result.warning }}</div>
    </div>
    {% elif error %}
    <div class="result-box">
        <div class="result-title danger">出错了</div>
        <div class="info">{{ error }}</div>
    </div>
    {% endif %}
</body>
</html>
"""

@app.route('/')
def index():
    result, error = do_recognize()
    return render_template_string(HTML_TEMPLATE, result=result, error=error)

if __name__ == '__main__':
    print("🚀 超简化AI识别服务已启动！")
    print("📱 访问地址：http://127.0.0.1:8099")
    # 自动获取局域网IP
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(("8.8.8.8", 80))
    local_ip = s.getsockname()[0]
    s.close()
    print(f"🔗 ESP32跳转地址：http://{local_ip}:8099")
    print(f"⚠️  请把上面的地址填到ESP32代码的AI_PAGE_URL配置里")
    
    from waitress import serve
    serve(app, host='0.0.0.0', port=8099)

🚀 超简化AI识别服务已启动！
📱 访问地址：http://127.0.0.1:8099
🔗 ESP32跳转地址：http://192.168.77.172:8099
⚠️  请把上面的地址填到ESP32代码的AI_PAGE_URL配置里


In [ ]:
import tensorflow as tf
import numpy as np
import os
import cv2
from flask import Flask, render_template_string

app = Flask(__name__)

# 只保留核心配置，先确保路径正确
PHOTOS_FOLDER = r"D:\bishe\esp32_photos_auto"
MODEL_PATH = r"D:\86151\bishedaima\disease_model_optimized.h5"

# 极简首页，先确保网页能打开
HTML_TEST = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>测试页面</title>
</head>
<body>
    <h1>✅ 网页服务正常！</h1>
    <p>如果能看到这个页面，说明Flask服务没问题</p>
    <p>模型路径：{{ model_path }}</p>
    <p>照片文件夹：{{ photo_path }}</p>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEST, 
                                 model_path=MODEL_PATH,
                                 photo_path=PHOTOS_FOLDER)

if __name__ == '__main__':
    # 改用Flask自带的开发服务器，更容易排查错误
    print("🚀 测试服务启动中...")
    print(f"访问地址：http://127.0.0.1:8099")
    app.run(host='0.0.0.0', port=9000, debug=True)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
import numpy as np
import os
import cv2
import re
import matplotlib.pyplot as plt
import base64
from io import BytesIO
from flask import Flask, render_template_string

# 解决中文显示乱码
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 初始化Flask
app = Flask(__name__)

# -------------------------- 核心配置（务必确认路径正确） --------------------------
PHOTOS_FOLDER = r"D:\bishe\esp32_photos_auto"
MODEL_PATH = r"D:\86151\bishedaima\disease_model_optimized.h5"
CONFIDENCE_THRESHOLD = 60
IMAGE_SIZE = (128, 128)
CLASS_NAMES_CN = [
    '番茄细菌性斑点病', '番茄早疫病', '番茄健康叶片',
    '番茄晚疫病', '番茄叶霉病', '番茄壳针孢叶斑病',
    '番茄二斑叶螨虫害', '番茄靶斑病', '番茄花叶病毒病',
    '番茄黄化曲叶病毒病'
]

# -------------------------- 病害防治建议 --------------------------
DISEASE_SUGGESTION = {
    '番茄细菌性斑点病': '建议及时摘除病叶，喷施噻唑锌、春雷霉素或氢氧化铜等细菌性杀菌剂，7-10天一次，连喷2-3次；注意通风降湿，避免叶片结露。',
    '番茄早疫病': '建议喷施代森锰锌、苯醚甲环唑或嘧菌酯等杀菌剂，及时摘除下部病叶；加强通风，控制棚内湿度，增施磷钾肥提升植株抗性。',
    '番茄健康叶片': '叶片健康无病害，建议保持当前环境管理，定期巡查监测，做好水肥管理和通风工作。',
    '番茄晚疫病': '建议立即摘除病叶病果，喷施霜霉威、氟啶胺或烯酰吗啉等专用药剂，5-7天一次，连喷2-3次；严格控制棚内湿度，避免大水漫灌。',
    '番茄叶霉病': '建议喷施氟硅唑、腈菌唑或甲基硫菌灵等药剂，加强通风排湿，适当降低种植密度，及时清理老叶病叶。',
    '番茄壳针孢叶斑病': '建议摘除病叶后喷施苯醚甲环唑、代森锰锌或百菌清，增施有机肥提升植株抗性，避免田间积水。',
    '番茄二斑叶螨虫害': '建议喷施阿维菌素、乙螨唑或螺螨酯等杀螨剂，重点喷洒叶片背面；注意轮换用药，避免害螨产生抗药性。',
    '番茄靶斑病': '建议喷施啶酰菌胺、氟唑菌酰胺或苯醚甲环唑，及时清除病残体，加强通风降湿，避免叶面长时间湿润。',
    '番茄花叶病毒病': '建议及时拔除病株，重点防治蚜虫、粉虱等传毒媒介，喷施盐酸吗啉胍、氨基寡糖素等抗病毒药剂；加强水肥管理，提升植株抗逆性。',
    '番茄黄化曲叶病毒病': '建议立即拔除病株，严防烟粉虱传播，喷施噻虫嗪、螺虫乙酯杀灭传毒害虫，配合氨基寡糖素提升植株抗病毒能力。'
}

# -------------------------- 生成概率分布图 --------------------------
def generate_confidence_chart(pred_prob, pred_class_idx):
    plt.figure(figsize=(10, 6), dpi=100)
    bars = plt.barh(CLASS_NAMES_CN, pred_prob * 100, color='#1f77b4')
    bars[pred_class_idx].set_color('#d62728')
    plt.xlabel('置信度（%）', fontsize=10)
    plt.xlim(0, 100)
    plt.title('10个病害类别的预测概率分布', fontsize=12, weight='bold')
    plt.tick_params(axis='y', labelsize=8)
    plt.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    
    # 转成base64嵌入网页
    buffer = BytesIO()
    plt.savefig(buffer, format='jpg', bbox_inches='tight')
    buffer.seek(0)
    img_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    plt.close()
    return img_base64

# -------------------------- 核心AI识别逻辑 --------------------------
def do_recognize():
    # 1. 检查文件夹
    if not os.path.exists(PHOTOS_FOLDER):
        return None, f"照片文件夹不存在：{PHOTOS_FOLDER}"
    
    # 2. 找最新照片
    image_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    all_image_files = [
        os.path.join(PHOTOS_FOLDER, filename)
        for filename in os.listdir(PHOTOS_FOLDER)
        if filename.endswith(image_extensions)
    ]
    
    if len(all_image_files) == 0:
        return None, f"照片文件夹里没有图片：{PHOTOS_FOLDER}"
    
    # 按时间排序取最新
    all_image_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    latest_image_path = all_image_files[0]
    image_name = os.path.basename(latest_image_path)

    # 3. 提取文件名里的环境参数
    env_params = {"temp": "无", "humi": "无", "soil": "无"}
    try:
        temp_match = re.search(r'temp(\d+\.?\d*)', image_name)
        humi_match = re.search(r'hum(\d+\.?\d*)', image_name)
        soil_match = re.search(r'soil(\d+\.?\d*)', image_name)
        if temp_match:
            env_params["temp"] = f"{temp_match.group(1)} ℃"
        if humi_match:
            env_params["humi"] = f"{humi_match.group(1)} %RH"
        if soil_match:
            env_params["soil"] = f"{soil_match.group(1)} %"
    except:
        pass

    # 4. 读取图片
    img_bgr = cv2.imread(latest_image_path)
    if img_bgr is None:
        from PIL import Image
        try:
            img_original = Image.open(latest_image_path).convert('RGB')
            img_rgb = np.array(img_original)
        except:
            return None, "图片文件损坏，无法读取"
    else:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 原始图片转base64
    _, img_original_buffer = cv2.imencode('.jpg', cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    img_original_base64 = base64.b64encode(img_original_buffer).decode('utf-8')

    # 6. 图片预处理
    img_resized = cv2.resize(img_rgb, IMAGE_SIZE)
    img_normalized = img_resized / 255.0
    
    # 7. 加载AI模型
    try:
        model = tf.keras.models.load_model(MODEL_PATH)
    except Exception as e:
        return None, f"模型加载失败：{str(e)}"
    
    # 8. AI预测
    img_input = np.expand_dims(img_normalized, axis=0)
    pred_prob = model.predict(img_input, verbose=0)[0]
    pred_class_idx = np.argmax(pred_prob)
    pred_class_name = CLASS_NAMES_CN[pred_class_idx]
    pred_confidence = round(float(pred_prob[pred_class_idx] * 100), 2)

    # 9. 生成概率分布图
    chart_base64 = generate_confidence_chart(pred_prob, pred_class_idx)

    # 10. 结果判断
    if pred_confidence < CONFIDENCE_THRESHOLD:
        result = {
            "status": "unknown",
            "name": image_name,
            "env_params": env_params,
            "class": "非番茄叶片/无法识别",
            "confidence": pred_confidence,
            "warning": "置信度不足，无法准确识别，请拍摄清晰的番茄叶片图片重试",
            "suggestion": "请确保叶片铺满画面、光照均匀、对焦清晰，重新拍摄后重试",
            "img_original": img_original_base64,
            "chart": chart_base64
        }
    else:
        result = {
            "status": "success",
            "name": image_name,
            "env_params": env_params,
            "class": pred_class_name,
            "confidence": pred_confidence,
            "is_healthy": pred_class_name == '番茄健康叶片',
            "warning": "叶片健康，无需处理" if pred_class_name == '番茄健康叶片' else f"检测到【{pred_class_name}】，请及时采取防治措施",
            "suggestion": DISEASE_SUGGESTION[pred_class_name],
            "img_original": img_original_base64,
            "chart": chart_base64
        }
    return result, None

# -------------------------- 网页模板 --------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>番茄叶片病虫害AI识别系统</title>
    <style>
        * {margin:0;padding:0;box-sizing:border-box; font-family: "Microsoft YaHei", sans-serif;}
        body {background: #f5f5f5; padding: 20px; max-width: 1000px; margin: 0 auto;}
        .header {text-align: center; margin-bottom: 30px;}
        .header h1 {color: #2c3e50; font-size: 28px;}
        .btn {
            background: #27ae60; color: white; border: none; padding: 12px 40px;
            font-size: 18px; border-radius: 8px; cursor: pointer; margin: 0 auto; display: block;
        }
        .btn:hover {background: #219653;}
        .card {
            background: white; border-radius: 12px; padding: 20px; margin-top: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }
        .card-title {font-size: 18px; font-weight: bold; color: #2c3e50; margin-bottom: 15px;}
        .result-title {
            font-size: 24px; text-align: center; margin: 20px 0;
            color: {% if result.is_healthy %}#27ae60{% else %}#e74c3c{% endif %};
        }
        .env-info {
            display: flex; gap: 20px; flex-wrap: wrap; margin: 20px 0;
        }
        .env-item {
            flex: 1; min-width: 150px; background: #f8f9fa; padding: 15px; border-radius: 8px;
            text-align: center;
        }
        .env-item .label {font-size: 14px; color: #7f8c8d;}
        .env-item .value {font-size: 20px; font-weight: bold; color: #2c3e50;}
        .img-box {text-align: center; margin: 20px 0;}
        .img-box img {max-width: 100%; max-height: 400px; border-radius: 8px;}
        .suggestion {
            background: #f8f9fa; border-left: 4px solid #3498db;
            padding: 15px; margin: 20px 0; border-radius: 4px;
        }
        .error {
            background: #fef0f0; color: #e74c3c; padding: 15px;
            border-radius: 8px; text-align: center; margin: 20px 0;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>番茄叶片病虫害AI识别系统</h1>
        <button class="btn" onclick="window.location.reload()">🔄 重新识别最新照片</button>
    </div>

    {% if error %}
    <div class="error">{{ error }}</div>
    {% else %}
    <div class="card">
        <div class="card-title">📷 拍摄的原始照片</div>
        <div class="img-box">
            <img src="data:image/jpeg;base64,{{ result.img_original }}" alt="原始照片">
            <p style="margin-top: 10px; color: #7f8c8d;">文件名：{{ result.name }}</p>
        </div>
    </div>

    <div class="card">
        <div class="card-title">🌡️ 采集时环境参数</div>
        <div class="env-info">
            <div class="env-item">
                <div class="label">环境温度</div>
                <div class="value">{{ result.env_params.temp }}</div>
            </div>
            <div class="env-item">
                <div class="label">环境湿度</div>
                <div class="value">{{ result.env_params.humi }}</div>
            </div>
            <div class="env-item">
                <div class="label">土壤湿度</div>
                <div class="value">{{ result.env_params.soil }}</div>
            </div>
        </div>
    </div>

    <div class="card">
        <div class="result-title">
            {{ result.class }} | 置信度：{{ result.confidence }}%
        </div>
        <div class="suggestion">
            <strong>📋 防治建议：</strong>{{ result.suggestion }}
        </div>
    </div>

    <div class="card">
        <div class="card-title">📊 病害类别概率分布</div>
        <div class="img-box">
            <img src="data:image/jpeg;base64,{{ result.chart }}" alt="概率分布图">
        </div>
    </div>
    {% endif %}
</body>
</html>
"""

# -------------------------- 网页路由 --------------------------
@app.route('/')
def index():
    result, error = do_recognize()
    return render_template_string(HTML_TEMPLATE, result=result, error=error)

# -------------------------- Jupyter专用启动配置 --------------------------
if __name__ == '__main__':
    print("="*50)
    print("🚀 番茄病虫害AI识别服务已启动！")
    print("📱 本地访问地址：http://127.0.0.1:9000")
    print("🔗 局域网访问地址：http://[你电脑的IP]:9000")
    print("⚠️  停止服务：点击Jupyter的停止按钮即可")
    print("="*50)
    # 【Jupyter专用】绝对不能开debug和reloader，否则必卡死
    app.run(host='0.0.0.0', port=9000, debug=False, use_reloader=False)

🚀 番茄病虫害AI识别服务已启动！
📱 本地访问地址：http://127.0.0.1:9000
🔗 局域网访问地址：http://[你电脑的IP]:9000
⚠️  停止服务：点击Jupyter的停止按钮即可
 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xba in position 2: invalid start byte

In [ ]:
import tensorflow as tf
import numpy as np
import os
import cv2
import base64
import re
from flask import Flask, render_template_string, request, jsonify

# 初始化Flask
app = Flask(__name__)

# -------------------------- 完全复用你的原有配置 --------------------------
PHOTOS_FOLDER = r"D:\bishe\esp32_photos_auto"
MODEL_PATH = r"D:\86151\bishedaima\disease_model_optimized.h5"
CONFIDENCE_THRESHOLD = 60  # 优化后的置信度阈值
IMAGE_SIZE = (128, 128)
CLASS_NAMES_CN = [
    '番茄细菌性斑点病', '番茄早疫病', '番茄健康叶片',
    '番茄晚疫病', '番茄叶霉病', '番茄壳针孢叶斑病',
    '番茄二斑叶螨虫害', '番茄靶斑病', '番茄花叶病毒病',
    '番茄黄化曲叶病毒病'
]

# -------------------------- 病害防治建议库（毕设加分项） --------------------------
DISEASE_SUGGESTION = {
    '番茄细菌性斑点病': '建议及时摘除病叶，喷施噻唑锌、春雷霉素或氢氧化铜等细菌性杀菌剂，7-10天一次，连喷2-3次；注意通风降湿，避免叶片结露。',
    '番茄早疫病': '建议喷施代森锰锌、苯醚甲环唑或嘧菌酯等杀菌剂，及时摘除下部病叶；加强通风，控制棚内湿度，增施磷钾肥提升植株抗性。',
    '番茄健康叶片': '叶片健康无病害，建议保持当前环境管理，定期巡查监测，做好水肥管理和通风工作。',
    '番茄晚疫病': '建议立即摘除病叶病果，喷施霜霉威、氟啶胺或烯酰吗啉等专用药剂，5-7天一次，连喷2-3次；严格控制棚内湿度，避免大水漫灌。',
    '番茄叶霉病': '建议喷施氟硅唑、腈菌唑或甲基硫菌灵等药剂，加强通风排湿，适当降低种植密度，及时清理老叶病叶。',
    '番茄壳针孢叶斑病': '建议摘除病叶后喷施苯醚甲环唑、代森锰锌或百菌清，增施有机肥提升植株抗性，避免田间积水。',
    '番茄二斑叶螨虫害': '建议喷施阿维菌素、乙螨唑或螺螨酯等杀螨剂，重点喷洒叶片背面；注意轮换用药，避免害螨产生抗药性。',
    '番茄靶斑病': '建议喷施啶酰菌胺、氟唑菌酰胺或苯醚甲环唑，及时清除病残体，加强通风降湿，避免叶面长时间湿润。',
    '番茄花叶病毒病': '建议及时拔除病株，重点防治蚜虫、粉虱等传毒媒介，喷施盐酸吗啉胍、氨基寡糖素等抗病毒药剂；加强水肥管理，提升植株抗逆性。',
    '番茄黄化曲叶病毒病': '建议立即拔除病株，严防烟粉虱传播，喷施噻虫嗪、螺虫乙酯杀灭传毒害虫，配合氨基寡糖素提升植株抗病毒能力。'
}

# -------------------------- 核心识别逻辑（复用你的代码，扩展返回全量数据） --------------------------
def do_recognize():
    # 1. 自动找到最新的照片
    if not os.path.exists(PHOTOS_FOLDER):
        return None, "文件夹不存在，请先创建文件夹"
    
    image_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    all_image_files = [
        os.path.join(PHOTOS_FOLDER, filename)
        for filename in os.listdir(PHOTOS_FOLDER)
        if filename.endswith(image_extensions)
    ]
    
    if len(all_image_files) == 0:
        return None, "文件夹里没有照片，请先从ESP32下载照片"
    
    # 取最新的照片
    all_image_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    latest_image_path = all_image_files[0]
    image_name = os.path.basename(latest_image_path)

    # 2. 从文件名提取环境参数（毕设亮点：环境和病害关联）
    env_params = {"temp": "无", "humi": "无", "soil": "无"}
    try:
        # 匹配文件名里的 tempXX_humXX_soilXX 格式
        temp_match = re.search(r'temp(\d+\.?\d*)', image_name)
        humi_match = re.search(r'hum(\d+\.?\d*)', image_name)
        soil_match = re.search(r'soil(\d+\.?\d*)', image_name)
        if temp_match:
            env_params["temp"] = f"{temp_match.group(1)} ℃"
        if humi_match:
            env_params["humi"] = f"{humi_match.group(1)} %RH"
        if soil_match:
            env_params["soil"] = f"{soil_match.group(1)} %"
    except:
        pass

    # 3. 读取图片+格式修复
    img_bgr = cv2.imread(latest_image_path)
    if img_bgr is None:
        from PIL import Image
        try:
            img_original = Image.open(latest_image_path).convert('RGB')
            img_rgb = np.array(img_original)
        except:
            return None, "图片文件损坏，无法读取"
    else:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 4. 图片预处理
    img_resized = cv2.resize(img_rgb, IMAGE_SIZE)
    img_normalized = img_resized / 255.0

    # 5. 原始图片转base64，用于网页展示
    _, img_original_buffer = cv2.imencode('.jpg', cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    img_original_base64 = base64.b64encode(img_original_buffer).decode('utf-8')

    # 6. 预处理后的图片转base64，用于网页展示
    _, img_preprocessed_buffer = cv2.imencode('.jpg', cv2.cvtColor((img_normalized * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
    img_preprocessed_base64 = base64.b64encode(img_preprocessed_buffer).decode('utf-8')

    # 7. AI模型预测
    model = tf.keras.models.load_model(MODEL_PATH)
    img_input = np.expand_dims(img_normalized, axis=0)
    pred_prob = model.predict(img_input, verbose=0)[0]
    # 所有类别的置信度（转成百分比，保留2位小数）
    all_confidence = [round(float(p * 100), 2) for p in pred_prob]
    # 最高置信度结果
    pred_class_idx = np.argmax(pred_prob)
    pred_class_name = CLASS_NAMES_CN[pred_class_idx]
    pred_confidence = all_confidence[pred_class_idx]

    # 8. 结果判断
    if pred_confidence < CONFIDENCE_THRESHOLD:
        result = {
            "status": "unknown",
            "name": image_name,
            "env_params": env_params,
            "class": "非番茄叶片/无法识别",
            "confidence": pred_confidence,
            "all_confidence": all_confidence,
            "class_names": CLASS_NAMES_CN,
            "warning": "置信度不足，无法准确识别，请拍摄清晰的番茄叶片图片重试",
            "suggestion": "请确保叶片铺满画面、光照均匀、对焦清晰，重新拍摄后重试",
            "img_original": img_original_base64,
            "img_preprocessed": img_preprocessed_base64
        }
    else:
        result = {
            "status": "success",
            "name": image_name,
            "env_params": env_params,
            "class": pred_class_name,
            "confidence": pred_confidence,
            "all_confidence": all_confidence,
            "class_names": CLASS_NAMES_CN,
            "is_healthy": pred_class_name == '番茄健康叶片',
            "warning": "叶片健康，无需处理" if pred_class_name == '番茄健康叶片' else f"检测到【{pred_class_name}】，请及时采取防治措施",
            "suggestion": DISEASE_SUGGESTION[pred_class_name],
            "img_original": img_original_base64,
            "img_preprocessed": img_preprocessed_base64
        }
    return result, None

# -------------------------- 网页路由 --------------------------
# 首页
@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

# 异步识别接口
@app.route('/api/recognize', methods=['GET'])
def api_recognize():
    result, error = do_recognize()
    if error:
        return jsonify({"status": "error", "msg": error}), 400
    return jsonify(result)

# -------------------------- 专业级网页模板（毕设级UI） --------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>智能苗圃番茄叶片病虫害AI预警系统</title>
    <!-- 引入ECharts用于概率柱状图 -->
    <script src="https://cdn.bootcdn.net/ajax/libs/echarts/5.4.3/echarts.min.js"></script>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: "Microsoft YaHei", "SimHei", sans-serif;
        }
        body {
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            min-height: 100vh;
            padding: 30px 20px;
        }
        .container {
            max-width: 1200px;
            margin: 0 auto;
        }
        .header {
            text-align: center;
            margin-bottom: 40px;
        }
        .header h1 {
            font-size: 36px;
            color: #2c3e50;
            margin-bottom: 10px;
            background: linear-gradient(90deg, #27ae60, #2980b9);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }
        .header p {
            color: #7f8c8d;
            font-size: 16px;
        }
        .control-panel {
            background: #fff;
            border-radius: 16px;
            padding: 30px;
            box-shadow: 0 8px 32px rgba(0,0,0,0.1);
            margin-bottom: 30px;
            text-align: center;
        }
        .recognize-btn {
            background: linear-gradient(90deg, #27ae60, #2ecc71);
            color: white;
            border: none;
            padding: 16px 60px;
            font-size: 20px;
            font-weight: bold;
            border-radius: 50px;
            cursor: pointer;
            box-shadow: 0 4px 15px rgba(39, 174, 96, 0.3);
            transition: all 0.3s ease;
        }
        .recognize-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 6px 20px rgba(39, 174, 96, 0.4);
        }
        .recognize-btn:disabled {
            background: #95a5a6;
            cursor: not-allowed;
            transform: none;
            box-shadow: none;
        }
        .loading {
            display: none;
            margin-top: 20px;
            color: #2980b9;
            font-size: 16px;
        }
        .error-box {
            background: #fef0f0;
            border: 1px solid #fde2e2;
            border-radius: 12px;
            padding: 20px;
            margin-top: 20px;
            color: #c0392b;
            text-align: center;
            font-size: 16px;
        }
        .result-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
            margin-bottom: 30px;
        }
        .card {
            background: #fff;
            border-radius: 16px;
            padding: 25px;
            box-shadow: 0 8px 32px rgba(0,0,0,0.1);
        }
        .card-title {
            font-size: 18px;
            font-weight: bold;
            color: #2c3e50;
            margin-bottom: 20px;
            padding-bottom: 10px;
            border-bottom: 2px solid #ecf0f1;
        }
        .img-box {
            text-align: center;
        }
        .img-box img {
            max-width: 100%;
            max-height: 300px;
            border-radius: 8px;
            border: 2px solid #ecf0f1;
        }
        .img-box p {
            margin-top: 10px;
            color: #7f8c8d;
            font-size: 14px;
        }
        .env-info {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            gap: 15px;
            margin-top: 20px;
        }
        .env-item {
            background: #f8f9fa;
            padding: 15px;
            border-radius: 8px;
            text-align: center;
        }
        .env-item .label {
            font-size: 14px;
            color: #7f8c8d;
            margin-bottom: 5px;
        }
        .env-item .value {
            font-size: 20px;
            font-weight: bold;
            color: #2c3e50;
        }
        .result-main {
            grid-column: 1 / -1;
            text-align: center;
        }
        .result-title {
            font-size: 28px;
            font-weight: bold;
            margin-bottom: 15px;
        }
        .result-healthy {
            color: #27ae60;
        }
        .result-danger {
            color: #c0392b;
        }
        .result-unknown {
            color: #7f8c8d;
        }
        .result-desc {
            font-size: 16px;
            color: #34495e;
            margin-bottom: 10px;
        }
        .suggestion-box {
            background: #f8f9fa;
            border-left: 4px solid #2980b9;
            padding: 20px;
            border-radius: 8px;
            margin-top: 20px;
            text-align: left;
        }
        .suggestion-box .title {
            font-weight: bold;
            color: #2980b9;
            margin-bottom: 10px;
        }
        .chart-box {
            grid-column: 1 / -1;
            height: 400px;
        }
        @media (max-width: 768px) {
            .result-grid {
                grid-template-columns: 1fr;
            }
            .header h1 {
                font-size: 24px;
            }
            .env-info {
                grid-template-columns: 1fr;
            }
            .chart-box {
                height: 500px;
            }
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>智能苗圃番茄叶片病虫害AI预警系统</h1>
            <p>基于深度学习的物联网智能苗圃病害监测系统</p>
        </div>

        <div class="control-panel">
            <button class="recognize-btn" id="recognizeBtn" onclick="startRecognize()">
                🔄 点击识别最新照片
            </button>
            <div class="loading" id="loading">
                正在加载模型、执行AI识别中，请稍候...
            </div>
            <div class="error-box" id="errorBox" style="display: none;"></div>
        </div>

        <div id="resultContainer" style="display: none;">
            <div class="result-grid">
                <!-- 原始图片 -->
                <div class="card">
                    <div class="card-title">📷 ESP32拍摄原始图片</div>
                    <div class="img-box">
                        <img id="originalImg" alt="原始图片">
                        <p id="imageName"></p>
                    </div>
                </div>

                <!-- 预处理后的图片 -->
                <div class="card">
                    <div class="card-title">🔧 AI预处理后的图片</div>
                    <div class="img-box">
                        <img id="preprocessedImg" alt="预处理图片">
                        <p>输入模型尺寸：128×128</p>
                    </div>
                </div>

                <!-- 环境参数 -->
                <div class="card">
                    <div class="card-title">🌡️ 采集时环境参数</div>
                    <div class="env-info">
                        <div class="env-item">
                            <div class="label">环境温度</div>
                            <div class="value" id="envTemp">--</div>
                        </div>
                        <div class="env-item">
                            <div class="label">环境湿度</div>
                            <div class="value" id="envHumi">--</div>
                        </div>
                        <div class="env-item">
                            <div class="label">土壤湿度</div>
                            <div class="value" id="envSoil">--</div>
                        </div>
                    </div>
                </div>

                <!-- 核心识别结果 -->
                <div class="card result-main">
                    <div class="result-title" id="resultTitle"></div>
                    <div class="result-desc" id="resultDesc"></div>
                    <div class="suggestion-box">
                        <div class="title">📋 智能防治建议</div>
                        <p id="suggestionText"></p>
                    </div>
                </div>

                <!-- 概率分布柱状图 -->
                <div class="card chart-box">
                    <div class="card-title">📊 10个病害类别预测概率分布</div>
                    <div id="confidenceChart" style="width: 100%; height: 100%;"></div>
                </div>
            </div>
        </div>
    </div>

    <script>
        // 初始化ECharts图表
        const chartDom = document.getElementById('confidenceChart');
        const myChart = echarts.init(chartDom);
        let chartOption;

        // 识别函数
        async function startRecognize() {
            const btn = document.getElementById('recognizeBtn');
            const loading = document.getElementById('loading');
            const errorBox = document.getElementById('errorBox');
            const resultContainer = document.getElementById('resultContainer');

            // 按钮禁用，显示加载
            btn.disabled = true;
            loading.style.display = 'block';
            errorBox.style.display = 'none';
            resultContainer.style.display = 'none';

            try {
                // 请求识别接口
                const res = await fetch('/api/recognize');
                const data = await res.json();

                if (data.status === 'error') {
                    throw new Error(data.msg);
                }

                // 填充数据
                document.getElementById('originalImg').src = 'data:image/jpeg;base64,' + data.img_original;
                document.getElementById('preprocessedImg').src = 'data:image/jpeg;base64,' + data.img_preprocessed;
                document.getElementById('imageName').textContent = '文件名：' + data.name;
                document.getElementById('envTemp').textContent = data.env_params.temp;
                document.getElementById('envHumi').textContent = data.env_params.humi;
                document.getElementById('envSoil').textContent = data.env_params.soil;

                // 识别结果样式
                const resultTitle = document.getElementById('resultTitle');
                const resultDesc = document.getElementById('resultDesc');
                const suggestionText = document.getElementById('suggestionText');

                if (data.status === 'unknown') {
                    resultTitle.className = 'result-title result-unknown';
                    resultTitle.textContent = `无法识别 | 最高置信度：${data.confidence}%`;
                    resultDesc.textContent = data.warning;
                } else {
                    if (data.is_healthy) {
                        resultTitle.className = 'result-title result-healthy';
                    } else {
                        resultTitle.className = 'result-title result-danger';
                    }
                    resultTitle.textContent = `识别结果：${data.class} | 置信度：${data.confidence}%`;
                    resultDesc.textContent = data.warning;
                }
                suggestionText.textContent = data.suggestion;

                // 渲染概率柱状图
                chartOption = {
                    tooltip: {
                        trigger: 'axis',
                        axisPointer: { type: 'shadow' },
                        formatter: '{b}: {c}%'
                    },
                    grid: { left: '15%', right: '5%', bottom: '5%', top: '5%' },
                    xAxis: { type: 'value', max: 100, name: '置信度（%）' },
                    yAxis: { 
                        type: 'category', 
                        data: data.class_names,
                        axisLabel: { fontSize: 12 }
                    },
                    series: [
                        {
                            type: 'bar',
                            data: data.all_confidence.map((val, idx) => {
                                return {
                                    value: val,
                                    itemStyle: {
                                        color: idx === data.class_names.indexOf(data.class) ? '#c0392b' : '#2980b9'
                                    }
                                }
                            })
                        }
                    ]
                };
                myChart.setOption(chartOption);

                // 显示结果
                resultContainer.style.display = 'block';

            } catch (e) {
                errorBox.style.display = 'block';
                errorBox.textContent = e.message;
            } finally {
                // 恢复按钮
                btn.disabled = false;
                loading.style.display = 'none';
            }
        }

        // 窗口大小变化时自适应图表
        window.addEventListener('resize', () => {
            myChart.resize();
        });
    </script>
</body>
</html>
"""

# -------------------------- 服务启动 --------------------------
if __name__ == '__main__':
    print("🚀 智能苗圃AI病虫害识别服务已启动！")
    print("📱 本地访问地址：http://127.0.0.1:8099")
    # 自动获取局域网IP
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(("8.8.8.8", 80))
    local_ip = s.getsockname()[0]
    s.close()
    print(f"🔗 ESP32跳转地址：http://{local_ip}:8099")
    print(f"⚠️  请把上面的地址填到ESP32代码的AI_PAGE_URL配置里")
    
    from waitress import serve
    serve(app, host='0.0.0.0', port=8099)

🚀 智能苗圃AI病虫害识别服务已启动！
📱 本地访问地址：http://127.0.0.1:8099
🔗 ESP32跳转地址：http://192.168.77.172:8099
⚠️  请把上面的地址填到ESP32代码的AI_PAGE_URL配置里
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
